# Phase 0: GCR Baseline Reproduction

**Goal:** Reproduce GCR on WebQSP within 1-2% of reported Hits@1 = 92.6

**Compute:** Requires A100 (40GB) — run this on Colab with A100 runtime.

**What this does:** Runs `gcr/workflow/predict_paths_and_answers.py` —
GCR's Step 1: graph-constrained path generation + answer extraction.
Hits@1 is computed from the extracted answers directly.

## 1. Verify Environment

In [ ]:
import torch, sys, os, json, warnings, gc, time, glob
import itertools, collections, random, math, re

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

IN_COLAB = 'google.colab' in sys.modules
print(f"In Colab: {IN_COLAB}")

assert torch.cuda.is_available(), "GPU required for Phase 0!"
vram = torch.cuda.get_device_properties(0).total_mem / 1e9
assert vram >= 30, f"Need >=30GB VRAM, have {vram:.1f}GB"

## 2. Clone This Repo and Install Dependencies

In [ ]:
# Clone this repo (which has GCR vendored in gcr/)
REPO_URL = "https://github.com/YOUR_USERNAME/dca-trie.git"

if not os.path.exists("dca-trie"):
    !git clone {REPO_URL}
%cd dca-trie
print(f"Working in: {os.getcwd()}")

In [ ]:
# Install this package (registers gcr/ and dca_trie/ as importable)
!pip install -e . --no-deps 2>&1 | tail -3

# Install heavy deps that Colab might not have, or might need specific versions
# Colab comes with torch + transformers pre-installed, but we need GCR's versions
!pip install -q transformers==4.44 accelerate>=0.30 peft>=0.11 \
    datasets>=2.19 sentence-transformers scikit-learn>=1.5 \
    marisa-trie python-dotenv tiktoken 2>&1 | tail -5

# Install flash-attention for speed (Colab has CUDA 12.1+)
try:
    !pip install -q flash-attn --no-build-isolation 2>&1 | tail -3
except:
    print("flash-attn install failed (non-critical), will use eager attention")

print("\nDependencies installed.")

In [ ]:
# Verify the imports work
from gcr.src.trie import MarisaTrie
from gcr.src.llms import get_registed_model
from gcr.src.utils.graph_utils import build_graph, dfs
from gcr.src.qa_prompt_builder import PathGenerationWithAnswerPromptBuilder
from dca_trie.semantic_scorer import SemanticScorer
print("All imports OK.")

## 3. HuggingFace Authentication

The GCR model (`rmanluo/GCR-Meta-Llama-3.1-8B-Instruct`) is gated.
You need:
1. A HuggingFace token: https://huggingface.co/settings/tokens
2. Access granted to: https://huggingface.co/rmanluo/GCR-Meta-Llama-3.1-8B-Instruct

Set the token below.

In [ ]:
from getpass import getpass

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    HF_TOKEN = getpass("Enter your HuggingFace token: ")

os.environ["HF_TOKEN"] = HF_TOKEN
with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=True)
print("HF_TOKEN configured.")

## 4. Snapshot Package Versions

Save for reproducibility section in thesis.

In [ ]:
!pip freeze > requirements_reproduced.txt
!wc -l requirements_reproduced.txt
print("Saved requirements_reproduced.txt")

## 5. Step 1: Graph-Constrained Decoding

First run on a **100-question subset** to validate.

The file `gcr/workflow/predict_paths_and_answers.py` is GCR's main
entry point for step 1. Key arguments:
- `--split test[:100]` — question subset (remove [:100] for full)
- `--k 10` — number of beams
- `--index_path_length 2` — max path depth
- `--generation_mode group-beam` — diverse beam search

In [ ]:
%%time
!python gcr/workflow/predict_paths_and_answers.py \
    --data_path rmanluo \
    --d RoG-webqsp \
    --split test[:100] \
    --index_path_length 2 \
    --model_name GCR-Meta-Llama-3.1-8B-Instruct \
    --model_path rmanluo/GCR-Meta-Llama-3.1-8B-Instruct \
    --k 10 \
    --prompt_mode zero-shot \
    --generation_mode group-beam \
    --attn_implementation flash_attention_2 \
    --n 1

### Check Results

In [ ]:
# Find the predictions file
result_pattern = "results/GenPaths/RoG-webqsp/GCR-Meta-Llama-3.1-8B-Instruct/test/*/predictions.jsonl"
result_dirs = glob.glob(result_pattern)
print(f"Found {len(result_dirs)} result files:")
for f in sorted(result_dirs):
    print(f"  {f}")

In [ ]:
# Show a sample prediction
if result_dirs:
    !head -3 "{result_dirs[0]}" | python -m json.tool 2>/dev/null | head -30
else:
    print("No results yet — run the cell above first.")

In [ ]:
# Check Hits@1 and F1
eval_files = glob.glob(
    "results/GenPaths/RoG-webqsp/GCR-Meta-Llama-3.1-8B-Instruct/test/*/eval_result*.txt"
)
if eval_files:
    for f in sorted(eval_files):
        print(f"\n=== {os.path.basename(f)} ===")
        with open(f) as fh:
            print(fh.read())
    print("\nTarget: Hits@1 >= 91%")
else:
    print("No eval results yet.")

**If Hits@1 < 91%:**
- Check the model path is correct (`rmanluo/GCR-Meta-Llama-3.1-8B-Instruct`, not the base Llama)
- Verify flash-attention is installed (speeds up inference)
- Check for OOM errors in the output above

**If the pipeline works, run the full test set:**

In [ ]:
# Run on full WebQSP test set — ~1-2 hours on A100
# Uncomment and run after validating on the 100-question subset:

# !python gcr/workflow/predict_paths_and_answers.py \
#     --data_path rmanluo \
#     --d RoG-webqsp \
#     --split test \
#     --index_path_length 2 \
#     --model_name GCR-Meta-Llama-3.1-8B-Instruct \
#     --model_path rmanluo/GCR-Meta-Llama-3.1-8B-Instruct \
#     --k 10 \
#     --prompt_mode zero-shot \
#     --generation_mode group-beam \
#     --attn_implementation flash_attention_2

## 6. Step 2 (Optional): Graph Inductive Reasoning

Uses GPT-4o-mini to reason over generated paths. Requires OpenAI API key.
The thesis exit criterion only requires Step 1 (Hits@1 is computed there).

In [ ]:
# Set your OpenAI key (uncomment):
# from getpass import getpass
# os.environ['OPENAI_API_KEY'] = getpass('OpenAI API key: ')

# Find the predictions file from Step 1:
# predictions_file = result_dirs[0] if result_dirs else "results/GenPaths/.../predictions.jsonl"

# !python gcr/workflow/predict_final_answer.py \
#     --data_path rmanluo \
#     --d RoG-webqsp \
#     --split test \
#     --model_name gpt-4o-mini \
#     --reasoning_path {predictions_file} \
#     --add_path True \
#     -n 10

## 7. Phase 0 Exit Criteria

- [ ] WebQSP Hits@1 >= 91% on 100-question subset
- [ ] Model correctly loaded (GCR fine-tuned, not base Llama)
- [ ] GPU memory usage stable (~35-40 GB)
- [ ] Output format matches `predictions.jsonl` spec
- [ ] `requirements_reproduced.txt` saved

Once passed, you're ready for **Phase 1: SIR Measurement**.